In [2]:
# %% [markdown]
# # 5 嵌入向量
# ## 5.1 理论计算题：Skip-gram负采样损失函数
# 符号：v_c(中心词向量), u_o(上下文词向量), u_{n_k}(第k个负样本向量)，K个负样本
# sigmoid σ(z) = 1/(1+e^{-z})
# 单样本对数似然目标（最大化）：
# log σ(v_c · u_o) + Σ_{k=1}^K log σ(-v_c · u_{n_k})
# 损失（最小化负对数似然）：
# L = - [ log σ(v_c^T u_o) + Σ_{k=1}^K log σ(-v_c^T u_{n_k}) ]
# 负样本采样：从噪声分布 P_n(w) 采样（常用词频3/4次方平滑分布），避开真实上下文词
#
# ## 5.2 编程题：CBOW完整softmax前向+交叉熵损失
# %%
import numpy as np

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

def cbow_forward_loss(context_ids, W, W_out, target_idx):
    """
    context_ids: list[int] 单个样本上下文词索引
    W: (V, d) 输入嵌入矩阵
    W_out: (d, V) 输出权重
    target_idx: int 中心词索引
    return: 标量交叉熵损失
    """
    # 取上下文向量并平均
    context_vecs = W[context_ids]  # (context_size, d)
    h = np.mean(context_vecs, axis=0)  # (d,)
    # 计算得分 logits
    logits = h @ W_out  # (V,)
    # softmax概率
    prob = softmax(logits)
    # 交叉熵损失 -log(p(target))
    loss = -np.log(prob[target_idx] + 1e-10)
    return loss

# 测试
if __name__ == "__main__":
    V, d = 10, 3
    W = np.random.randn(V, d)
    W_out = np.random.randn(d, V)
    ctx = [0, 2, 5]
    tgt = 3
    loss_val = cbow_forward_loss(ctx, W, W_out, tgt)
    print("CBOW 交叉熵损失值：", loss_val)

CBOW 交叉熵损失值： 2.5354018712539936
